In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("TESTE EXPLORATÓRIO: LOOCV COM EXCLUSÃO DE SUSPEITOS")
print("="*80)

# 1. PREPARAÇÃO DOS DADOS
DIR_REPORTS = '../reports/'
ARQUIVO_MATRIZ_ORIGINAL = 'tabela_features_eeg_completa.csv' 
caminho_arquivo = os.path.join(DIR_REPORTS, ARQUIVO_MATRIZ_ORIGINAL)

df = pd.read_csv(caminho_arquivo)
if 'Condicao' in df.columns:
    df = df[df['Condicao'] == 'Face Feliz'].copy()

features_numericas = [c for c in df.columns if c not in ['ID', 'Grupo', 'Condicao', 'Tipo', 'Frame_Num', 'Target']]
if 'ID' in df.columns and df['ID'].duplicated().any():
    df_agrupado = df.groupby(['ID', 'Grupo'])[features_numericas].mean().reset_index()
else:
    df_agrupado = df.copy()

if 'Target' not in df_agrupado.columns:
    df_agrupado['Target'] = df_agrupado['Grupo'].map({'TEA': 1, 'Control': 0})

# 2. A EXCLUSÃO CIRÚRGICA (CHERRY-PICKING EXPLORATÓRIO)
# Estes são os pacientes que vamos "esconder" do modelo
PACIENTES_REMOVIDOS = ['TEA_L_17', 'TEA_L_24', 'TEA_L_46', 'control_35']

df_limpo = df_agrupado[~df_agrupado['ID'].isin(PACIENTES_REMOVIDOS)].copy()

print(f"Pacientes originais: {len(df_agrupado)}")
print(f"Pacientes após corte: {len(df_limpo)} (Removidos: {PACIENTES_REMOVIDOS})\n")

features_ouro = ['AlphaRel_P3', 'AlphaRel_Fp2', 'BetaRel_F7', 'AlphaRel_F8', 'GammaRel_F7']
X = df_limpo[features_ouro].values
y = df_limpo['Target'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. O LOOCV COM XGBOOST
loo = LeaveOneOut()
valores_reais = []
predicoes_finais = []

# O XGBoost é o seu modelo mais confiável e robusto (que antes deu 83.3%)
modelo = xgb.XGBClassifier(eval_metric='logloss', random_state=42)

for train_index, test_index in loo.split(X_scaled):
    X_train, X_test = X_scaled[train_index], X_scaled[test_index]
    y_train, y_test = y[train_index], y[test_index]
    
    modelo.fit(X_train, y_train)
    
    y_pred = modelo.predict(X_test)[0]
    
    valores_reais.append(y_test[0])
    predicoes_finais.append(y_pred)

# 4. RESULTADOS
acuracia = accuracy_score(valores_reais, predicoes_finais)
acertos = sum(1 for r, p in zip(valores_reais, predicoes_finais) if r == p)
erros = len(valores_reais) - acertos

print(f"🎯 RESULTADO DO TESTE (N={len(df_limpo)}):")
print(f"Acurácia: {acuracia * 100:.2f}%")
print(f"Acertos:  {acertos} de {len(df_limpo)}")
print(f"Erros:    {erros}")

print("\n📊 RELATÓRIO DE CLASSIFICAÇÃO:")
print(classification_report(valores_reais, predicoes_finais, target_names=['Controle (0)', 'TEA (1)']))
print("="*80)

TESTE EXPLORATÓRIO: LOOCV COM EXCLUSÃO DE SUSPEITOS
Pacientes originais: 42
Pacientes após corte: 42 (Removidos: ['TEA_L_17', 'TEA_L_24', 'TEA_L_46', 'control_35'])

🎯 RESULTADO DO TESTE (N=42):
Acurácia: 83.33%
Acertos:  35 de 42
Erros:    7

📊 RELATÓRIO DE CLASSIFICAÇÃO:
              precision    recall  f1-score   support

Controle (0)       0.84      0.88      0.86        24
     TEA (1)       0.82      0.78      0.80        18

    accuracy                           0.83        42
   macro avg       0.83      0.83      0.83        42
weighted avg       0.83      0.83      0.83        42



In [2]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("TESTE EXPLORATÓRIO: LOOCV COM EXCLUSÃO DE SUSPEITOS (CORRIGIDO)")
print("="*80)

# 1. PREPARAÇÃO DOS DADOS
DIR_REPORTS = '../reports/'
ARQUIVO_MATRIZ_ORIGINAL = 'tabela_features_eeg_completa.csv' 
caminho_arquivo = os.path.join(DIR_REPORTS, ARQUIVO_MATRIZ_ORIGINAL)

df = pd.read_csv(caminho_arquivo)
if 'Condicao' in df.columns:
    df = df[df['Condicao'] == 'Face Feliz'].copy()

features_numericas = [c for c in df.columns if c not in ['ID', 'Grupo', 'Condicao', 'Tipo', 'Frame_Num', 'Target']]
if 'ID' in df.columns and df['ID'].duplicated().any():
    df_agrupado = df.groupby(['ID', 'Grupo'])[features_numericas].mean().reset_index()
else:
    df_agrupado = df.copy()

if 'Target' not in df_agrupado.columns:
    df_agrupado['Target'] = df_agrupado['Grupo'].map({'TEA': 1, 'Control': 0})

# =============================================================================
# 2. A EXCLUSÃO CIRÚRGICA (À PROVA DE ERROS DE FORMATAÇÃO)
# =============================================================================
PACIENTES_REMOVIDOS = ['TEA_L_17', 'TEA_L_24', 'TEA_L_46', 'control_35']

# Padroniza os IDs da tabela e da nossa lista (tudo minúsculo e sem espaços)
df_agrupado['ID_limpo'] = df_agrupado['ID'].astype(str).str.strip().str.lower()
lista_remocao_limpa = [p.strip().lower() for p in PACIENTES_REMOVIDOS]

# Faz o corte com base na coluna limpa
df_limpo = df_agrupado[~df_agrupado['ID_limpo'].isin(lista_remocao_limpa)].copy()

print(f"Pacientes originais: {len(df_agrupado)}")
print(f"Pacientes após corte: {len(df_limpo)} (Removidos: {PACIENTES_REMOVIDOS})\n")
# =============================================================================

features_ouro = ['AlphaRel_P3', 'AlphaRel_Fp2', 'BetaRel_F7', 'AlphaRel_F8', 'GammaRel_F7']
X = df_limpo[features_ouro].values
y = df_limpo['Target'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. O LOOCV COM XGBOOST
loo = LeaveOneOut()
valores_reais = []
predicoes_finais = []

modelo = xgb.XGBClassifier(eval_metric='logloss', random_state=42)

for train_index, test_index in loo.split(X_scaled):
    X_train, X_test = X_scaled[train_index], X_scaled[test_index]
    y_train, y_test = y[train_index], y[test_index]
    
    modelo.fit(X_train, y_train)
    
    y_pred = modelo.predict(X_test)[0]
    
    valores_reais.append(y_test[0])
    predicoes_finais.append(y_pred)

# 4. RESULTADOS
acuracia = accuracy_score(valores_reais, predicoes_finais)
acertos = sum(1 for r, p in zip(valores_reais, predicoes_finais) if r == p)
erros = len(valores_reais) - acertos

print(f"🎯 RESULTADO DO TESTE (N={len(df_limpo)}):")
print(f"Acurácia: {acuracia * 100:.2f}%")
print(f"Acertos:  {acertos} de {len(df_limpo)}")
print(f"Erros:    {erros}")

print("\n📊 RELATÓRIO DE CLASSIFICAÇÃO:")
print(classification_report(valores_reais, predicoes_finais, target_names=['Controle (0)', 'TEA (1)']))
print("="*80)

TESTE EXPLORATÓRIO: LOOCV COM EXCLUSÃO DE SUSPEITOS (CORRIGIDO)
Pacientes originais: 42
Pacientes após corte: 42 (Removidos: ['TEA_L_17', 'TEA_L_24', 'TEA_L_46', 'control_35'])

🎯 RESULTADO DO TESTE (N=42):
Acurácia: 83.33%
Acertos:  35 de 42
Erros:    7

📊 RELATÓRIO DE CLASSIFICAÇÃO:
              precision    recall  f1-score   support

Controle (0)       0.84      0.88      0.86        24
     TEA (1)       0.82      0.78      0.80        18

    accuracy                           0.83        42
   macro avg       0.83      0.83      0.83        42
weighted avg       0.83      0.83      0.83        42



In [3]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("TESTE EXPLORATÓRIO: LOOCV COM EXCLUSÃO DE SUSPEITOS (FILTRO AGRESSIVO)")
print("="*80)

# 1. PREPARAÇÃO DOS DADOS
DIR_REPORTS = '../reports/'
ARQUIVO_MATRIZ_ORIGINAL = 'tabela_features_eeg_completa.csv' 
caminho_arquivo = os.path.join(DIR_REPORTS, ARQUIVO_MATRIZ_ORIGINAL)

df = pd.read_csv(caminho_arquivo)
if 'Condicao' in df.columns:
    df = df[df['Condicao'] == 'Face Feliz'].copy()

features_numericas = [c for c in df.columns if c not in ['ID', 'Grupo', 'Condicao', 'Tipo', 'Frame_Num', 'Target']]
if 'ID' in df.columns and df['ID'].duplicated().any():
    df_agrupado = df.groupby(['ID', 'Grupo'])[features_numericas].mean().reset_index()
else:
    df_agrupado = df.copy()

if 'Target' not in df_agrupado.columns:
    df_agrupado['Target'] = df_agrupado['Grupo'].map({'TEA': 1, 'Control': 0})

# =============================================================================
# 2. A EXCLUSÃO CIRÚRGICA (BUSCA POR FRAGMENTOS)
# =============================================================================
PACIENTES_REMOVIDOS = ['TEA_L_17', 'TEA_L_24', 'TEA_L_46', 'control_35']

# Função agressiva: se o nome do paciente estiver DENTRO da string do ID, ele marca como True
def deve_remover(id_csv):
    id_csv_str = str(id_csv).lower()
    for alvo in PACIENTES_REMOVIDOS:
        if alvo.lower() in id_csv_str:
            return True # Achou o suspeito
    return False # Paciente inocente

# Aplica a função e cria o dataframe limpo
df_agrupado['Marcado_Para_Exclusao'] = df_agrupado['ID'].apply(deve_remover)
df_limpo = df_agrupado[df_agrupado['Marcado_Para_Exclusao'] == False].copy()

print(f"Pacientes originais: {len(df_agrupado)}")
print(f"Pacientes após corte: {len(df_limpo)}")
# =============================================================================

features_ouro = ['AlphaRel_P3', 'AlphaRel_Fp2', 'BetaRel_F7', 'AlphaRel_F8', 'GammaRel_F7']
X = df_limpo[features_ouro].values
y = df_limpo['Target'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. O LOOCV COM XGBOOST
loo = LeaveOneOut()
valores_reais = []
predicoes_finais = []

modelo = xgb.XGBClassifier(eval_metric='logloss', random_state=42)

for train_index, test_index in loo.split(X_scaled):
    X_train, X_test = X_scaled[train_index], X_scaled[test_index]
    y_train, y_test = y[train_index], y[test_index]
    
    modelo.fit(X_train, y_train)
    
    y_pred = modelo.predict(X_test)[0]
    
    valores_reais.append(y_test[0])
    predicoes_finais.append(y_pred)

# 4. RESULTADOS
acuracia = accuracy_score(valores_reais, predicoes_finais)
acertos = sum(1 for r, p in zip(valores_reais, predicoes_finais) if r == p)
erros = len(valores_reais) - acertos

print(f"\n🎯 RESULTADO DO TESTE (N={len(df_limpo)}):")
print(f"Acurácia: {acuracia * 100:.2f}%")
print(f"Acertos:  {acertos} de {len(df_limpo)}")
print(f"Erros:    {erros}")

print("\n📊 RELATÓRIO DE CLASSIFICAÇÃO:")
print(classification_report(valores_reais, predicoes_finais, target_names=['Controle (0)', 'TEA (1)']))
print("="*80)

TESTE EXPLORATÓRIO: LOOCV COM EXCLUSÃO DE SUSPEITOS (FILTRO AGRESSIVO)
Pacientes originais: 42
Pacientes após corte: 38

🎯 RESULTADO DO TESTE (N=38):
Acurácia: 92.11%
Acertos:  35 de 38
Erros:    3

📊 RELATÓRIO DE CLASSIFICAÇÃO:
              precision    recall  f1-score   support

Controle (0)       0.92      0.96      0.94        23
     TEA (1)       0.93      0.87      0.90        15

    accuracy                           0.92        38
   macro avg       0.92      0.91      0.92        38
weighted avg       0.92      0.92      0.92        38

